# P2-A4 — RAG End-to-End (retrieve + generate)

Assembly time. You have all three pieces: **prompting** (P2-A1), **structured/grounded calls**, and **semantic retrieval** (P2-A3). RAG = retrieve the relevant context, stuff it into the prompt, and let the LLM answer *from that context*.

The knowledge base below is about a **made-up company ("Zephyr Cloud")** — facts the base model cannot possibly know from training. That's deliberate: if the model answers correctly, it can *only* be because your retrieval worked. It's how you prove RAG is actually grounding the answer.

In [1]:
# --- Setup (given) ---
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
CHAT_MODEL = 'gpt-4o-mini'
EMBED_MODEL = 'text-embedding-3-small'

def embed(texts):
    if isinstance(texts, str):
        texts = [texts]
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([d.embedding for d in resp.data])

def ask(prompt, system=None, **kwargs):
    messages = []
    if system:
        messages.append({'role': 'system', 'content': system})
    messages.append({'role': 'user', 'content': prompt})
    resp = client.chat.completions.create(model=CHAT_MODEL, messages=messages, **kwargs)
    return resp.choices[0].message.content

# --- Given: knowledge base about a FICTIONAL company (model can't know these) ---
kb = [
    "Zephyr Cloud's free tier includes 5 GB of storage and 100 GB of bandwidth per month.",
    "To upgrade your Zephyr plan, go to Billing > Plans and choose Pro or Enterprise.",
    "Zephyr Cloud's Pro plan costs $29 per month and includes 1 TB of storage.",
    "Zephyr support is available 24/7 via in-app chat for Pro and Enterprise customers.",
    "Zephyr's data centers are located in Oregon, Frankfurt, and Singapore.",
]
kb_emb = embed(kb)   # embed the knowledge base once, up front
print('KB embedded:', kb_emb.shape)

KB embedded: (5, 1536)


## Task 1 — `retrieve(question, k)`
Write a function that embeds the question, cosine-compares it to `kb_emb`, and returns the **top-k** most relevant KB strings.
<br>This is your P2-A3 code wrapped in a function. Test it: `retrieve('how much is Pro?', k=2)` should surface the $29 fact.

In [2]:
# TODO: def retrieve(question, k=2): ... return the top-k kb strings

# def retrieve(question, k=2): ... return the top-k kb strings
def retrieve(question, k=2):
    q = embed(question)[0]
    sims = (kb_emb @ q) / (np.linalg.norm(kb_emb, axis=1) * np.linalg.norm(q))
    top = np.argsort(sims)[::-1][:k]
    return [kb[i] for i in top]

retrieve('how much is Pro?', k=2)


["Zephyr Cloud's Pro plan costs $29 per month and includes 1 TB of storage.",
 'To upgrade your Zephyr plan, go to Billing > Plans and choose Pro or Enterprise.']

## Task 2 — `answer(question)` = retrieve + generate
Write the full RAG function: call `retrieve()`, build a prompt that includes the retrieved context, and instruct the model to **answer ONLY from that context — and say it doesn't know if the answer isn't there.** Return the model's answer.
<br>Hint: your prompt should look roughly like: `Context:\n{joined retrieved docs}\n\nQuestion: {question}\n\nAnswer using only the context above; if it's not there, say you don't know.`

In [3]:
# TODO: def answer(question): retrieve context, build grounded prompt, return ask(...)

# def answer(question): retrieve context, build grounded prompt, return ask(...)
def answer(question, k=2):
    context = "\n".join(retrieve(question, k))
    prompt = (
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\n"
        "Answer using ONLY the context above. "
        "If the answer is not in the context, say you don't know."
    )
    return ask(prompt)


## Task 3 — Test grounding (the moment of truth)
Call `answer()` on two questions and print both:
1. **Answerable** from the KB — e.g. *"How much does the Pro plan cost and how much storage does it include?"* → should give the right facts.
2. **NOT in the KB** — e.g. *"Does Zephyr have a data center in Tokyo?"* → the model should say it doesn't know / it's not in the context (NOT make something up).
<br>Goal: prove RAG both *answers from your data* and *refuses to hallucinate* when the data isn't there.

In [4]:
# TODO: test answer() on an in-KB question and an out-of-KB question


# test answer() on an in-KB question and an out-of-KB question
q1 = "How much does the Pro plan cost and how much storage does it include?"
q2 = "Does Zephyr have a data center in Tokyo?"

print('Q1 (answerable):', q1)
print('A1:', answer(q1))
print()
print('Q2 (not in KB):', q2)
print('A2:', answer(q2))


Q1 (answerable): How much does the Pro plan cost and how much storage does it include?
A1: The Pro plan costs $29 per month and includes 1 TB of storage.

Q2 (not in KB): Does Zephyr have a data center in Tokyo?
A2: I don't know.


## Task 4 — Explain (in your own words)
1. Why does RAG let a model correctly answer about 'Zephyr Cloud' when it was never trained on it? What problem does that solve for real apps?
2. What is the single biggest failure point of this pipeline — i.e. if the final answer is wrong, where did it most likely break, and how would you measure that (tie it back to evals, W2-A2)?

> _your answer here_

1. The base model has no Zephyr Cloud facts in its weights — that company is fictional and post-dates nothing it ever saw. RAG sidesteps that by injecting the relevant facts into the prompt at query time: we retrieve the matching KB lines and the model just reads and reasons over text placed in front of it, which it's very good at. For real apps this solves two problems at once — it lets the model answer over private/proprietary/up-to-date data it was never trained on, and it grounds answers in a known source so you can trust (and cite) them instead of relying on the model's fuzzy memory. No retraining or fine-tuning needed; just update the knowledge base.

2. The biggest failure point is retrieval, not generation. If the top-k chunks handed to the model don't actually contain the answer, the model can't produce it from thin air — it either says "I don't know" (best case) or hallucinates (worst case). So when a final answer is wrong, it most often broke because the right chunk wasn't retrieved (bad embeddings match, k too small, KB chunked badly). I'd measure it exactly like W2-A2 evals: build a set of question→expected-answer pairs, and score retrieval separately from generation — e.g. "was the gold chunk in the top-k?" (retrieval recall) and "is the final answer correct / does it correctly abstain on out-of-KB questions?" (answer accuracy / hallucination rate). Tracking those two numbers separately tells you which half to fix.